In [1]:
import pandas as pd

df = pd.read_csv('Master Query Table - Sheet1 (1).csv', na_values=['\\N', ''])
print(df.shape)
df.head()

(244747, 31)


,txnId,referralID,txnPeriod,txnAmount,txnFee,discount,isCashbackUsed,cashback,txnType,paymentType,...,transferDestinationBank,depositBank,disco,powerProvider,airtimeTelco,dataTelco,satTvProvider,customerAvgAmountSoFar,deviationFromCustomerAvg,txnCountLast24h
0,106417,a1,2024-09-24 06:17:34,600.0,0.0,0.0,0,0.0,Commission,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,101321,a10,2024-09-23 23:57:00,600.0,0.0,0.0,0,0.0,Commission,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,74809,a12,2024-09-23 10:36:01,600.0,0.0,0.0,0,0.0,Commission,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,31952,a13,2024-01-29 22:10:56,600.0,0.0,0.0,0,0.0,Commission,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,33161,a13,2024-01-29 23:30:12,500.0,0.0,0.0,1,0.0,Airtime Recharge,User Wallet,...,NaN,NaN,NaN,NaN,MTN,NaN,NaN,600.0,0.833333,1


In [2]:

df['txnPeriod'] = pd.to_datetime(df['txnPeriod'])
df['signupDate'] = pd.to_datetime(df['signupDate'])


df = df.sort_values('txnPeriod').reset_index(drop=True)

print(df['txnPeriod'].min(), "to", df['txnPeriod'].max())

2023-01-10 00:00:00 to 2026-03-25 10:16:53


In [3]:

df['txnFee'] = df['txnFee'].fillna(0)
df['kycLevel'] = df['kycLevel'].fillna(0) 
df['deviationFromCustomerAvg'] = df['deviationFromCustomerAvg'].fillna(1)  
df['customerAvgAmountSoFar'] = df['customerAvgAmountSoFar'].fillna(df['txnAmount'])


cat_cols = ['paymentType', 'txnIPAddress', 'txnPlatform', 'txnLocation', 'gender',
            'transferDestinationBank', 'depositBank', 'disco', 'powerProvider',
            'airtimeTelco', 'dataTelco', 'satTvProvider']
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN')

df.isna().sum().sum()  

np.int64(0)

In [4]:

df = df.sort_values(['referralID', 'txnType', 'txnPeriod'])

df['customerTypeAvgSoFar'] = (
    df.groupby(['referralID', 'txnType'])['txnAmount']
      .transform(lambda x: x.expanding().mean().shift(1))
)

df['deviationFromCustomerTypeAvg'] = df['txnAmount'] / df['customerTypeAvgSoFar']
df['deviationFromCustomerTypeAvg'] = df['deviationFromCustomerTypeAvg'].fillna(1).clip(upper=20)


df = df.sort_values('txnPeriod').reset_index(drop=True)

df[['referralID', 'txnType', 'txnAmount', 'customerTypeAvgSoFar', 'deviationFromCustomerTypeAvg']].head(10)

,referralID,txnType,txnAmount,customerTypeAvgSoFar,deviationFromCustomerTypeAvg
0,lexer,Wallet Transfer - Out,2000.0,NaN,1.0
1,ayafal,Bank Transfer,2000.0,NaN,1.0
2,lexer,Electricity Recharge,1900.0,NaN,1.0
3,ayafal,Wallet Transfer - Out,2000.0,NaN,1.0
4,ayafal,Electricity Recharge,500.0,NaN,1.0
5,ayafal,Electricity Recharge,500.0,500.0,1.0
6,ayafal,Electricity Recharge,500.0,500.0,1.0
7,ayafal,Electricity Recharge,500.0,500.0,1.0
8,ayafal,Electricity Recharge,500.0,500.0,1.0
9,ayafal,Electricity Recharge,500.0,500.0,1.0


In [5]:
import numpy as np


if 'transferAccountNo' not in df.columns:
    recipient_data = pd.read_csv('transferAccountNo_export.csv', na_values=['\\N', ''])
    df = df.merge(recipient_data, on='txnId', how='left')


has_comma = df['txnLocation'].str.contains(',', na=False)
split_raw = df.loc[has_comma, 'txnLocation'].str.split(',', expand=True)

def clean_to_float(series):
    cleaned = series.str.replace(r'[^0-9\.\-]', '', regex=True)
    return pd.to_numeric(cleaned, errors='coerce')

df['txnLat'] = np.nan
df['txnLon'] = np.nan
df.loc[has_comma, 'txnLat'] = clean_to_float(split_raw[0])
df.loc[has_comma, 'txnLon'] = clean_to_float(split_raw[1])

df = df.sort_values(['referralID', 'txnPeriod'])
df['prevLat'] = df.groupby('referralID')['txnLat'].shift(1)
df['prevLon'] = df.groupby('referralID')['txnLon'].shift(1)
df['prevTxnPeriod'] = df.groupby('referralID')['txnPeriod'].shift(1)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distanceFromPrevKm'] = haversine(df['prevLat'], df['prevLon'], df['txnLat'], df['txnLon'])
df['hoursSincePrevTxn'] = (df['txnPeriod'] - df['prevTxnPeriod']).dt.total_seconds() / 3600
df['impliedSpeedKmh'] = df['distanceFromPrevKm'] / df['hoursSincePrevTxn'].replace(0, np.nan)
df['impossible_travel'] = (df['impliedSpeedKmh'] > 900).astype(int)
df['new_country_flag'] = (df['distanceFromPrevKm'] > 1300).astype(int)


df['ip_seen_before'] = df.groupby(['referralID', 'txnIPAddress']).cumcount() > 0
df['new_ip_flag'] = (~df['ip_seen_before']).astype(int)


df['txnDate'] = df['txnPeriod'].dt.date
df['dailyCumulativeAmount'] = df.groupby(['referralID', 'txnDate'])['txnAmount'].transform('sum')
df['high_daily_amount_flag'] = (df['dailyCumulativeAmount'] > 500000).astype(int)


df['recipient_seen_before'] = df.groupby(['referralID', 'transferAccountNo'])['transferAccountNo'].transform(lambda x: x.duplicated())
df['recipient_seen_before'] = df['recipient_seen_before'].fillna(False).astype(bool)
df['first_time_recipient'] = ((~df['recipient_seen_before']) & df['transferAccountNo'].notna()).astype(int)
df['first_time_high_value'] = ((df['first_time_recipient'] == 1) & (df['txnAmount'] > 200000)).astype(int)


df['new_account_high_value'] = ((df['accountAgeDays'] <= 7) & (df['txnAmount'] > 200000)).astype(int)
df['new_account_high_velocity'] = ((df['accountAgeDays'] <= 7) & (df['txnCountLast24h'] >= 5)).astype(int)

df = df.sort_values('txnPeriod').reset_index(drop=True)


check_cols = ['impossible_travel','new_country_flag','new_ip_flag','high_daily_amount_flag',
              'first_time_recipient','first_time_high_value','new_account_high_value','new_account_high_velocity']
print(df[check_cols].sum())

impossible_travel              4225
new_country_flag                646
new_ip_flag                  107480
high_daily_amount_flag         8873
first_time_recipient          12114
first_time_high_value           160
new_account_high_value          113
new_account_high_velocity     30099
dtype: int64


In [6]:
df.to_csv('fraud_model_data_with_features.csv', index=False)
print("Saved.")

Saved.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score


if 'df' not in locals():
    print("Loading dataset from disk...")
    df = pd.read_csv('fraud_model_data_with_features.csv', parse_dates=['txnPeriod', 'signupDate'])


if 'coordinated_signup_flag_v2' not in df.columns:
    print("Computing missing engineered features...")
    
   
    real_ips = df[df['txnIPAddress'] != 'UNKNOWN']
    account_main_ip = real_ips.groupby('referralID')['txnIPAddress'].agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    signup_dates = df.drop_duplicates(subset='referralID').set_index('referralID')['signupDate']
    signup_by_ip2 = pd.DataFrame({'txnIPAddress': account_main_ip, 'signupDate': signup_dates}).dropna().reset_index()
    signup_by_ip2 = signup_by_ip2.sort_values(['txnIPAddress', 'signupDate'])

    def count_nearby_signups(group):
        dates = group['signupDate'].values
        counts = [((group['signupDate'] >= d - pd.Timedelta(hours=72)) & (group['signupDate'] <= d + pd.Timedelta(hours=72))).sum() - 1 for d in dates]
        group['nearby_signups_same_ip'] = counts
        return group

    signup_by_ip2 = signup_by_ip2.groupby('txnIPAddress', group_keys=False).apply(count_nearby_signups)
    nearby_map2 = signup_by_ip2.set_index('referralID')['nearby_signups_same_ip'].to_dict()
    df['coordinated_signup_flag_v2'] = df['referralID'].map(nearby_map2).fillna(0)

    
    df = df.sort_values(['referralID', 'txnPeriod']).set_index('txnPeriod')
    df['cumulativeSpend24h'] = df.groupby('referralID')['txnAmount'].rolling('24h').sum().reset_index(level=0, drop=True)
    df = df.reset_index()
    df['cumulativeSpend24h'] = df['cumulativeSpend24h'] - df['txnAmount']

    df['destBank_interaction_count'] = df.groupby(['referralID', 'transferDestinationBank']).cumcount()
    df['is_first_time_dest_bank'] = (df['destBank_interaction_count'] == 0).astype(int)
    df.loc[df['transferDestinationBank'] == 'UNKNOWN', 'is_first_time_dest_bank'] = 0

   
    df['impossible_travel_flag'] = df['impossible_travel']
    df = df.sort_values('txnPeriod').reset_index(drop=True)
    
   
    df.to_csv('fraud_model_data_with_features.csv', index=False)
    print("Saved all 14 features permanently to fraud_model_data_with_features.csv!")


block_list = pd.read_csv('block_list.csv', sep=';')
block_list['created_at'] = pd.to_datetime(block_list['created_at'])
named_entries = block_list.dropna(subset=['customer_name'])

ref = pd.read_csv('ReferralID_customers.csv')
matched = ref.merge(named_entries, left_on='customer', right_on='customer_name', how='inner')

name_counts = matched['customer_name'].value_counts()
safe_names = name_counts[name_counts == 1].index
safe_matches = matched[matched['customer_name'].isin(safe_names)]

matches_ring = ref[ref['customer'].astype(str).str.contains('Kitsume|Magisk', case=False, na=False)]
ring_ids = set(matches_ring['referralID'].tolist())
safe_blacklist_dates = safe_matches.set_index('referralID')['created_at'].to_dict()


df['blacklist_date'] = df['referralID'].map(safe_blacklist_dates)
is_ring_txn = df['referralID'].isin(ring_ids)
is_attack_window_txn = (
    df['blacklist_date'].notna() & 
    (df['txnPeriod'] >= (df['blacklist_date'] - pd.Timedelta(days=14))) & 
    (df['txnPeriod'] <= (df['blacklist_date'] + pd.Timedelta(days=1)))
)
df['is_confirmed_fraud'] = (is_ring_txn | is_attack_window_txn).astype(int)

print(f"Confirmed fraud accounts: {df[df['is_confirmed_fraud']==1]['referralID'].nunique()}")
print(f"Confirmed fraud transactions: {df['is_confirmed_fraud'].sum()}")


df['customerAvgAmountSoFar'] = df['customerTypeAvgSoFar'].fillna(df['txnAmount'])
df['deviationFromCustomerAvg'] = df['deviationFromCustomerTypeAvg'].fillna(1.0).clip(upper=50.0)
df['txnCountLast24h'] = df['txnCountLast24h'].clip(upper=100)

final_feature_list = [
    'txnAmount', 'txnFee', 'discount', 'isCashbackUsed', 'cashback', 
    'accountAgeDays', 'accountStatus', 'customerAvgAmountSoFar', 
    'deviationFromCustomerAvg', 'txnCountLast24h', 'coordinated_signup_flag_v2',
    'cumulativeSpend24h', 'is_first_time_dest_bank', 'impossible_travel_flag'
]
X_matrix = df[final_feature_list].fillna(0).copy()
y = df['is_confirmed_fraud']


sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(X_matrix, y, groups=df['referralID']))

X_train, X_test = X_matrix.iloc[train_idx], X_matrix.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train fraud: {y_train.sum()} txns ({df.iloc[train_idx][y_train==1]['referralID'].nunique()} accounts)")
print(f"Test fraud:  {y_test.sum()} txns ({df.iloc[test_idx][y_test==1]['referralID'].nunique()} accounts)")


imbalance_ratio = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
final_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.08,
    scale_pos_weight=imbalance_ratio,
    random_state=42
)
final_model.fit(X_train, y_train)


test_probs = final_model.predict_proba(X_test)[:, 1]
print("\n=== MODEL PERFORMANCE (UNSEEN ACCOUNTS TEST SET) ===")
print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, test_probs):.4f}")

print("\n=== THRESHOLD CALIBRATION ===")
for t in [0.30, 0.50, 0.70, 0.85, 0.95]:
    pred = (test_probs >= t).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    recall = tp / max((tp + fn), 1) * 100
    precision = tp / max((tp + fp), 1) * 100
    print(f"Threshold {t:.2f} -> Recall: {recall:5.1f}% ({tp}/{tp+fn}) | Precision: {precision:5.1f}% | FP: {fp}")


joblib.dump(final_model, "xgb_model_FINAL.pkl")
joblib.dump(final_feature_list, "final_cols_FINAL.pkl")

DEMO_DIR = "../moniflow_fraud_demo/moniflow_fraud_demo"
if os.path.exists(DEMO_DIR):
    joblib.dump(final_model, os.path.join(DEMO_DIR, "xgb_model_behavioral.pkl"))
    joblib.dump(final_feature_list, os.path.join(DEMO_DIR, "final_cols_behavioral.pkl"))
    
    demo_cols = [
        'txnId', 'referralID', 'txnPeriod', 'txnType', 'txnPlatform', 
        'txnIPAddress', 'transferAccountNo', 'kycLevel', 'is_confirmed_fraud'
    ] + final_feature_list
    df[demo_cols].to_csv(os.path.join(DEMO_DIR, "Cleaned_transactions_new.csv"), index=False)
    print("\nSynced model and enriched dataset directly to moniflow_fraud_demo!")


test_df = df.iloc[test_idx].copy()
test_df['risk_prob'] = test_probs
print("\n=== SAMPLE UNSEEN FRAUD TXN IDs FOR STREAMLIT DEMO ===")
print(test_df[test_df['is_confirmed_fraud'] == 1].sort_values('risk_prob', ascending=False)[['txnId', 'referralID', 'txnAmount', 'accountAgeDays', 'risk_prob']].head(5).to_string(index=False))

print("\n=== SAMPLE UNSEEN NORMAL TXN IDs FOR STREAMLIT DEMO ===")
print(test_df[test_df['is_confirmed_fraud'] == 0].sort_values('risk_prob', ascending=True)[['txnId', 'referralID', 'txnAmount', 'accountAgeDays', 'risk_prob']].head(5).to_string(index=False))

Computing missing engineered features...
Saved all 14 features permanently to fraud_model_data_with_features.csv!
Confirmed fraud accounts: 72
Confirmed fraud transactions: 532
Train fraud: 424 txns (57 accounts)
Test fraud:  108 txns (15 accounts)

=== MODEL PERFORMANCE (UNSEEN ACCOUNTS TEST SET) ===
ROC-AUC: 0.9687
PR-AUC:  0.5365

=== THRESHOLD CALIBRATION ===
Threshold 0.30 -> Recall:  82.4% (89/108) | Precision:  14.7% | FP: 518
Threshold 0.50 -> Recall:  79.6% (86/108) | Precision:  22.2% | FP: 301
Threshold 0.70 -> Recall:  78.7% (85/108) | Precision:  27.2% | FP: 228
Threshold 0.85 -> Recall:  77.8% (84/108) | Precision:  33.3% | FP: 168
Threshold 0.95 -> Recall:  76.9% (83/108) | Precision:  41.1% | FP: 119

Synced model and enriched dataset directly to moniflow_fraud_demo!

=== SAMPLE UNSEEN FRAUD TXN IDs FOR STREAMLIT DEMO ===
 txnId referralID  txnAmount  accountAgeDays  risk_prob
261268  magiskk26     7205.0               2   0.999375
261265  magiskk26     7224.0          

In [ ]:
import os
import numpy as np
import pandas as pd

DEMO_DIR = "../moniflow_fraud_demo/moniflow_fraud_demo"


new_batch = pd.read_csv("Cleaned_transactions_new.csv", na_values=['\\N', ''], parse_dates=['txnPeriod', 'signupDate'])
print(f"Loaded newer batch: {new_batch.shape}")
print(f"Is 289162 in newer batch? {289162 in new_batch['txnId'].values}")


common_raw_cols = [
    'txnId', 'referralID', 'txnPeriod', 'signupDate', 'txnAmount', 'txnFee', 
    'discount', 'isCashbackUsed', 'cashback', 'txnType', 'txnPlatform', 
    'txnIPAddress', 'txnLocation', 'transferDestinationBank', 'transferAccountNo', 
    'kycLevel', 'accountAgeDays', 'accountStatus', 'txnCountLast24h'
]

for col in common_raw_cols:
    if col not in new_batch.columns:
        new_batch[col] = np.nan
    if col not in df.columns:
        df[col] = np.nan

full_df = pd.concat([df[common_raw_cols], new_batch[common_raw_cols]], ignore_index=True)
full_df = full_df.drop_duplicates(subset=['txnId'], keep='first')
full_df = full_df.sort_values(['referralID', 'txnPeriod']).reset_index(drop=True)
print(f"Combined total transactions (2023 - Aug 2026): {len(full_df):,}")


full_df['txnFee'] = full_df['txnFee'].fillna(0)
full_df['discount'] = full_df['discount'].fillna(0)
full_df['cashback'] = full_df['cashback'].fillna(0)
full_df['isCashbackUsed'] = full_df['isCashbackUsed'].fillna(0)
full_df['kycLevel'] = full_df['kycLevel'].fillna(1)
full_df['accountStatus'] = full_df['accountStatus'].fillna(1)
full_df['txnIPAddress'] = full_df['txnIPAddress'].fillna('UNKNOWN')
full_df['transferDestinationBank'] = full_df['transferDestinationBank'].fillna('UNKNOWN')
full_df['txnLocation'] = full_df['txnLocation'].fillna('UNKNOWN')


full_df = full_df.sort_values(['referralID', 'txnType', 'txnPeriod'])
full_df['customerTypeAvgSoFar'] = (
    full_df.groupby(['referralID', 'txnType'])['txnAmount']
           .transform(lambda x: x.expanding().mean().shift(1))
)
full_df['customerAvgAmountSoFar'] = full_df['customerTypeAvgSoFar'].fillna(full_df['txnAmount'])
full_df['deviationFromCustomerAvg'] = (full_df['txnAmount'] / full_df['customerAvgAmountSoFar']).fillna(1.0).clip(upper=50.0)
full_df['txnCountLast24h'] = full_df['txnCountLast24h'].fillna(0).clip(upper=100)


real_ips = full_df[full_df['txnIPAddress'] != 'UNKNOWN']
account_main_ip = real_ips.groupby('referralID')['txnIPAddress'].agg(lambda x: x.mode()[0] if not x.mode().empty else None)
signup_dates = full_df.drop_duplicates(subset='referralID').set_index('referralID')['signupDate']
signup_by_ip2 = pd.DataFrame({'txnIPAddress': account_main_ip, 'signupDate': signup_dates}).dropna().reset_index()
signup_by_ip2 = signup_by_ip2.sort_values(['txnIPAddress', 'signupDate'])

def count_nearby_signups(group):
    dates = group['signupDate'].values
    counts = [((group['signupDate'] >= d - pd.Timedelta(hours=72)) & (group['signupDate'] <= d + pd.Timedelta(hours=72))).sum() - 1 for d in dates]
    group['nearby_signups_same_ip'] = counts
    return group

signup_by_ip2 = signup_by_ip2.groupby('txnIPAddress', group_keys=False).apply(count_nearby_signups)
nearby_map2 = signup_by_ip2.set_index('referralID')['nearby_signups_same_ip'].to_dict()
full_df['coordinated_signup_flag_v2'] = full_df['referralID'].map(nearby_map2).fillna(0)


full_df = full_df.sort_values(['referralID', 'txnPeriod']).set_index('txnPeriod')
full_df['cumulativeSpend24h'] = full_df.groupby('referralID')['txnAmount'].rolling('24h').sum().reset_index(level=0, drop=True)
full_df = full_df.reset_index()
full_df['cumulativeSpend24h'] = full_df['cumulativeSpend24h'] - full_df['txnAmount']

full_df['destBank_interaction_count'] = full_df.groupby(['referralID', 'transferDestinationBank']).cumcount()
full_df['is_first_time_dest_bank'] = (full_df['destBank_interaction_count'] == 0).astype(int)
full_df.loc[full_df['transferDestinationBank'] == 'UNKNOWN', 'is_first_time_dest_bank'] = 0


has_comma = full_df['txnLocation'].astype(str).str.contains(',', na=False)
split_raw = full_df.loc[has_comma, 'txnLocation'].astype(str).str.split(',', expand=True)
full_df['lat'] = np.nan
full_df['lon'] = np.nan
full_df.loc[has_comma, 'lat'] = pd.to_numeric(split_raw[0].str.replace(r'[^0-9\.\-]', '', regex=True), errors='coerce')
full_df.loc[has_comma, 'lon'] = pd.to_numeric(split_raw[1].str.replace(r'[^0-9\.\-]', '', regex=True), errors='coerce')

full_df = full_df.sort_values(['referralID', 'txnPeriod'])
full_df['prev_lat'] = full_df.groupby('referralID')['lat'].shift(1)
full_df['prev_lon'] = full_df.groupby('referralID')['lon'].shift(1)
full_df['prev_txnPeriod'] = full_df.groupby('referralID')['txnPeriod'].shift(1)
full_df['hours_since_last_txn'] = (full_df['txnPeriod'] - full_df['prev_txnPeriod']).dt.total_seconds() / 3600.0

R = 6371.0
lat1, lon1, lat2, lon2 = map(np.radians, [full_df['lat'], full_df['lon'], full_df['prev_lat'], full_df['prev_lon']])
a = np.sin((lat2 - lat1)/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2)**2
full_df['distance_km'] = 2 * R * np.arcsin(np.sqrt(a))
full_df['travel_speed_kmh'] = full_df['distance_km'] / full_df['hours_since_last_txn'].replace(0, np.nan)
full_df['impossible_travel_flag'] = ((full_df['travel_speed_kmh'] > 900) & (full_df['hours_since_last_txn'] < 24)).astype(int)


final_feature_list = [
    'txnAmount', 'txnFee', 'discount', 'isCashbackUsed', 'cashback', 
    'accountAgeDays', 'accountStatus', 'customerAvgAmountSoFar', 
    'deviationFromCustomerAvg', 'txnCountLast24h', 'coordinated_signup_flag_v2',
    'cumulativeSpend24h', 'is_first_time_dest_bank', 'impossible_travel_flag'
]
demo_cols = [
    'txnId', 'referralID', 'txnPeriod', 'txnType', 'txnPlatform', 
    'txnIPAddress', 'transferAccountNo', 'kycLevel'
] + final_feature_list


demo_cols = list(dict.fromkeys(demo_cols))
full_df[demo_cols].to_csv(os.path.join(DEMO_DIR, "Cleaned_transactions_new.csv"), index=False)
print(f"Saved all {len(full_df):,} transactions to {DEMO_DIR}/Cleaned_transactions_new.csv!")

Loaded newer batch: (35233, 32)
Is 289162 in newer batch? True
Combined total transactions (2023 - Aug 2026): 279,882
Saved all 279,882 transactions to ../moniflow_fraud_demo/moniflow_fraud_demo/Cleaned_transactions_new.csv!


In [ ]:
import os
import joblib
import pandas as pd


DEMO_DIR = "../moniflow_fraud_demo/moniflow_fraud_demo"
if not os.path.exists(DEMO_DIR):
    DEMO_DIR = "."


df_check = pd.read_csv(os.path.join(DEMO_DIR, "Cleaned_transactions_new.csv"), parse_dates=["txnPeriod"])
model = joblib.load(os.path.join(DEMO_DIR, "xgb_model_behavioral.pkl"))
final_cols = joblib.load(os.path.join(DEMO_DIR, "final_cols_behavioral.pkl"))


match = df_check[
    (df_check["referralID"] == "joeli") &
    (df_check["accountAgeDays"] == 540) &
    (df_check["txnAmount"] == 20000.0) &
    (df_check["txnIPAddress"] == "197.211.63.149")
]

if match.empty:
    print("Transaction not found! Check filter values.")
else:
    row = match.iloc[0]
    txn_id = row["txnId"]
    txn_time = row["txnPeriod"]
    print(f"✅ Found Transaction ID: {txn_id} | Timestamp: {txn_time}\n")

   
    joeli_all = df_check[df_check["referralID"] == "joeli"].sort_values("txnPeriod")

   
    prior_bank_txns = joeli_all[
        (joeli_all["txnType"] == "Bank Transfer") &
        (joeli_all["txnPeriod"] < txn_time)
    ]
    manual_avg = prior_bank_txns["txnAmount"].mean()
    manual_dev = row["txnAmount"] / manual_avg

    
    window_start = txn_time - pd.Timedelta(hours=24)
    prior_24h_txns = joeli_all[
        (joeli_all["txnPeriod"] >= window_start) &
        (joeli_all["txnPeriod"] < txn_time)
    ]
    manual_24h_spend = prior_24h_txns["txnAmount"].sum()
    manual_24h_count = len(prior_24h_txns)

    
    manual_ips = joeli_all[joeli_all["txnIPAddress"].notna() & (joeli_all["txnIPAddress"] != "UNKNOWN")]["txnIPAddress"].nunique()
    manual_recs = joeli_all["transferAccountNo"].dropna().nunique()

    
    print("=== DATA VERIFICATION (STORED CSV vs. MANUAL RECALCULATION) ===")
    print(f"Amount:                 CSV = ₦{row['txnAmount']:,.2f}")
    print(f"Customer Avg (Type):    CSV = ₦{row['customerAvgAmountSoFar']:,.2f}  | Manual Recalc = ₦{manual_avg:,.2f} (from {len(prior_bank_txns)} prior Bank Transfers)")
    print(f"Deviation:              CSV = {row['deviationFromCustomerAvg']:.2f}x       | Manual Recalc = {manual_dev:.2f}x")
    print(f"Prior 24h Spend:        CSV = ₦{row['cumulativeSpend24h']:,.2f} | Manual Recalc = ₦{manual_24h_spend:,.2f}")
    print(f"Prior 24h Txn Count:    CSV = {int(row['txnCountLast24h'])} txns      | Manual Recalc = {manual_24h_count} txns")
    print(f"Total Known IPs/Recs:   Manual Recalc = {manual_ips} IP(s) · {manual_recs} recipient(s)")

    
    X_row = pd.DataFrame([row[final_cols]])[final_cols]
    prob = float(model.predict_proba(X_row)[0][1]) * 100
    print(f"\n=== MODEL SCORE VERIFICATION ===")
    print(f"Fraud Risk Probability: {prob:.4f}% (App rounds to {prob:.2f}%)")

    try:
        import shap
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X_row)
        sv_row = sv[1][0] if isinstance(sv, list) else (sv.values[0] if hasattr(sv, "values") else sv[0])
        shap_df = pd.DataFrame({"Feature": final_cols, "Value": X_row.iloc[0].values, "SHAP_Impact": sv_row})
        shap_df["Abs"] = shap_df["SHAP_Impact"].abs()
        print("\n=== TOP 4 SHAP FACTORS ===")
        print(shap_df.sort_values("Abs", ascending=False)[["Feature", "Value", "SHAP_Impact"]].head(4).to_string(index=False))
    except ImportError:
        print("Install shap to verify SHAP numbers.")

✅ Found Transaction ID: 263386 | Timestamp: 2026-03-24 16:34:47

=== DATA VERIFICATION (STORED CSV vs. MANUAL RECALCULATION) ===
Amount:                 CSV = ₦20,000.00
Customer Avg (Type):    CSV = ₦9,075.08  | Manual Recalc = ₦9,075.08 (from 181 prior Bank Transfers)
Deviation:              CSV = 2.20x       | Manual Recalc = 2.20x
Prior 24h Spend:        CSV = ₦51,837.50 | Manual Recalc = ₦51,837.50
Prior 24h Txn Count:    CSV = 6 txns      | Manual Recalc = 6 txns
Total Known IPs/Recs:   Manual Recalc = 376 IP(s) · 46 recipient(s)

=== MODEL SCORE VERIFICATION ===
Fraud Risk Probability: 0.0098% (App rounds to 0.01%)


c:\Users\johnc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== TOP 4 SHAP FACTORS ===
                   Feature   Value  SHAP_Impact
            accountAgeDays   540.0    -5.425185
        cumulativeSpend24h 51837.5    -0.969997
                 txnAmount 20000.0    -0.922863
coordinated_signup_flag_v2     0.0    -0.791893


In [10]:
print(match[[
    "txnId", "txnPeriod", "txnAmount", 
    "customerAvgAmountSoFar", "txnCountLast24h", "cumulativeSpend24h"
]])

         txnId           txnPeriod  txnAmount  customerAvgAmountSoFar  \
155884  263386 2026-03-24 16:34:47    20000.0             9075.083260   
155891  263519 2026-03-25 08:35:09    20000.0             9088.716952   

        txnCountLast24h  cumulativeSpend24h  
155884                6             51837.5  
155891               10             98800.0  


In [11]:
row = match[match["txnCountLast24h"] == 10].iloc[0]

In [ ]:
import os
import pandas as pd

# 1. Automatically locate the full 279,882-row dataset (not the 35k partial batch)
possible_paths = [
    r"C:\Users\johnc\Downloads\moniflow_fraud_demo\moniflow_fraud_demo\Cleaned_transactions_new.csv",
    "fraud_model_data_with_features.csv",
    "transactions_fixed.csv",
    "Cleaned_transactions_new.csv",
]

df = None
for path in possible_paths:
    if os.path.exists(path):
        temp_df = pd.read_csv(path, low_memory=False)
        print(f"Checked: {path} -> {len(temp_df):,} rows")
        # Make sure we use the dataset that actually contains the pre-March 25 data (>200,000 rows)
        if len(temp_df) > 200_000:
            df = temp_df
            print(f"✅ Loaded Master Dataset: {path} ({len(df):,} rows)\n")
            break

if df is None:
    raise FileNotFoundError("Could not find the 279,882-row master CSV. Check the file path!")

# 2. Filter for all Magisk Kitsume syndicate accounts
kitsume_mask = df["referralID"].astype(str).str.lower().str.contains("magisk|kitsume", na=False)
kitsume_df = df[kitsume_mask].copy()

print("=" * 75)
print(f"1. TOTAL MAGISK KITSUME ACCOUNTS FOUND: {kitsume_df['referralID'].nunique()}")
print(f"   TOTAL TRANSACTIONS BY THESE ACCOUNTS: {len(kitsume_df)}")
print("=" * 75)
print("All Syndicate referralIDs:")
print(sorted(kitsume_df["referralID"].unique()))

# 3. Check the exact accounts sharing IP 102.91.96.152 with magiskk26
ip_target = "102.91.96.152"
same_ip_accts = df[df["txnIPAddress"] == ip_target]["referralID"].dropna().unique()

print("\n" + "=" * 75)
print(f"2. ACCOUNTS LINKED TO IP {ip_target} (Total: {len(same_ip_accts)}):")
print("=" * 75)
print(sorted(same_ip_accts))

# 4. Verify the ~N7,205 Bank Transfer cash-out signature across the syndicate
bank_transfers = kitsume_df[kitsume_df["txnType"] == "Bank Transfer"]

print("\n" + "=" * 75)
print("3. MAGISK KITSUME BANK TRANSFER CASH-OUT AMOUNTS (TOP 10 MOST COMMON):")
print("=" * 75)
print(
    bank_transfers["txnAmount"]
    .value_counts()
    .head(10)
    .rename_axis("Cash-Out Amount (Naira)")
    .reset_index(name="Number of Times Attempted")
    .to_string(index=False)
)

# 5. Show the step-by-step transaction sequence for sample bots
print("\n" + "=" * 75)
print("4. SAMPLE TRANSACTION SEQUENCE PER BOT ACCOUNT (WARM-UP -> N7,205 DRAIN):")
print("=" * 75)
sample_cols = [
    "txnId",
    "referralID",
    "txnPeriod",
    "txnType",
    "txnAmount",
    "txnIPAddress",
]
existing_cols = [c for c in sample_cols if c in kitsume_df.columns]

for bot_id in ["magiskk19", "magiskk26", "magiskk27"]:
    if bot_id in kitsume_df["referralID"].values:
        print(f"\n--- Account: {bot_id} ---")
        bot_txns = kitsume_df[kitsume_df["referralID"] == bot_id].sort_values("txnPeriod")
        print(bot_txns[existing_cols].to_string(index=False))   

Checked: C:\Users\johnc\Downloads\moniflow_fraud_demo\moniflow_fraud_demo\Cleaned_transactions_new.csv -> 279,882 rows
✅ Loaded Master Dataset: C:\Users\johnc\Downloads\moniflow_fraud_demo\moniflow_fraud_demo\Cleaned_transactions_new.csv (279,882 rows)

1. TOTAL MAGISK KITSUME ACCOUNTS FOUND: 54
   TOTAL TRANSACTIONS BY THESE ACCOUNTS: 406
All Syndicate referralIDs:
['kitsume01', 'kitsume99', 'kitsumek', 'kitsumekitsume', 'kitsumem', 'kitsumemagisk', 'kkitsume', 'kmagisk', 'magisk99', 'magisk_01', 'magiskk', 'magiskk10', 'magiskk11', 'magiskk13', 'magiskk18', 'magiskk19', 'magiskk23', 'magiskk25', 'magiskk26', 'magiskk27', 'magiskk29', 'magiskk30', 'magiskk32', 'magiskk35', 'magiskk352', 'magiskk37', 'magiskk39', 'magiskk42', 'magiskk447', 'magiskk50', 'magiskk515', 'magiskk52', 'magiskk520', 'magiskk54', 'magiskk55', 'magiskk59', 'magiskk60', 'magiskk61', 'magiskk65', 'magiskk66', 'magiskk745', 'magiskk75', 'magiskk77', 'magiskk78', 'magiskk82', 'magiskk85', 'magiskk87', 'magiskk95', 